# Simpson's paradox trong dữ liệu Olist thật

Khách ở bang SP có tỉ lệ review xấu thấp hơn khách ở các bang khác. "Khách SP dễ tính hơn"? Notebook
này xây công cụ để kiểm tra một so sánh tỉ lệ có đổi chiều khi phân tầng không, rồi áp lên Olist.

Định nghĩa dùng chung của track: đơn giao xong (`delivered` và có ngày khách nhận), review mới nhất
mỗi đơn, review xấu là `review_score <= 2`, giao trễ là so nguyên timestamp với ngày hẹn.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm và hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/09-eda-storytelling/exercises/intermediate/02-simpson-paradox -q`

In [ ]:
import numpy as np
import pandas as pd

DAY_BINS = [0, 7, 14, 21, 30, np.inf]
DAY_LABELS = ["0-6", "7-13", "14-20", "21-29", "30+"]

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")

## 1. Bảng phân tích mức đơn

In [ ]:
def build_orders(orders: pd.DataFrame, reviews: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi đơn giao xong có review. Cột: order_id, customer_state, late, bad, delivery_days.

    giao xong: order_status == "delivered" và có order_delivered_customer_date
    review: mỗi đơn giữ review có review_answer_timestamp mới nhất; bỏ đơn không có review
    late: order_delivered_customer_date > order_estimated_delivery_date (so nguyên timestamp)
    bad: review_score <= 2
    delivery_days: (order_delivered_customer_date − order_purchase_timestamp) tính bằng ngày (float)
    """
    # TODO
    raise NotImplementedError


def delivery_bucket(days: pd.Series) -> pd.Series:
    """Nhóm số ngày giao theo DAY_BINS (khoảng [a, b), trái đóng phải mở), nhãn DAY_LABELS, có thứ tự."""
    # TODO: pd.cut
    raise NotImplementedError

In [ ]:
# explore: kiểm tra nhanh bảng đơn
df = build_orders(orders, reviews, customers)
df["region"] = np.where(df.customer_state == "SP", "SP", "ngoài SP")
df["days"] = delivery_bucket(df.delivery_days)
print(len(df), df.late.mean().round(4), df.bad.mean().round(4))
df.groupby("region").bad.agg(["mean", "size"])

## 2. Tỉ lệ theo tầng, chuẩn hoá trực tiếp, và kiểm tra đảo chiều

Công thức chuẩn hoá: tỉ lệ chuẩn hoá của nhóm g là tổng theo tầng s của w_s nhân tỉ lệ của g ở tầng s, với cùng bộ trọng số w cho mọi nhóm.

In [ ]:
def stratified_rates(df: pd.DataFrame, group: str, stratum: str, target: str) -> pd.DataFrame:
    """Tỉ lệ target theo (stratum, group). Trả DataFrame cột [stratum, group, "n", "rate"] (tên cột
    stratum và group giữ nguyên tên gốc), sắp theo stratum rồi group; chỉ gồm cặp có dữ liệu."""
    # TODO: groupby([stratum, group], observed=True)
    raise NotImplementedError


def standardized_rates(df: pd.DataFrame, group: str, stratum: str, target: str,
                       weights: pd.Series | None = None) -> pd.Series:
    """Direct standardization: với mỗi group g, sum_s w_s * rate(g, s).

    weights: Series index là giá trị stratum. None → phân bố stratum của chính df (tỉ trọng số dòng).
    Chỉ dùng các stratum có mặt ở MỌI group (và có trong weights); chuẩn hoá lại trọng số của các
    stratum đó cho tổng bằng 1. Trả Series index là các group.
    """
    # TODO
    raise NotImplementedError


def simpson_check(df: pd.DataFrame, group: str, stratum: str, target: str, a, b) -> dict:
    """So sánh group a với group b (chỉ giữ hai group này trong df).

    crude_diff           rate(a) − rate(b) trên dữ liệu gộp
    adjusted_diff        standardized(a) − standardized(b), trọng số mặc định của standardized_rates
    n_strata             số stratum có mặt ở cả a và b
    strata_against_crude số stratum trong đó rate(a, s) − rate(b, s) khác 0 và ngược dấu crude_diff
    reversal             True nếu crude_diff và adjusted_diff đều khác 0 và ngược dấu
    """
    # TODO
    raise NotImplementedError

## 3. SP và các bang khác, chia theo số ngày giao

In [ ]:
# explore: bảng theo tầng và tỉ trọng mỗi tầng trong từng nhóm
rates = stratified_rates(df, "region", "days", "bad")
display(rates.pivot(index="days", columns="region", values="rate").round(4))
display(pd.crosstab(df.days, df.region, normalize="columns").round(3))
simpson_check(df, "region", "days", "bad", "SP", "ngoài SP")

In [ ]:
# explore: vẽ tỉ lệ theo tầng (thứ tự tự nhiên của bucket, không sắp theo giá trị)
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))
sns.pointplot(data=df, x="days", y="bad", hue="region", errorbar=("ci", 95), ax=ax)
ax.set_title("Cùng số ngày giao, khách SP không dễ tính hơn")
ax.set_xlabel("Số ngày từ lúc đặt tới lúc nhận")
ax.set_ylabel("Tỉ lệ review 1–2 sao")
plt.show()

## 4. Cùng dữ liệu, chia theo trễ / đúng hẹn

In [ ]:
# explore
display(stratified_rates(df, "region", "late", "bad"))
simpson_check(df, "region", "late", "bad", "SP", "ngoài SP")

## 5. Kết luận

Viết 5–8 câu: con số gộp nói gì, phân tầng theo số ngày giao nói gì, vì sao chúng khác nhau (dùng tỉ
trọng mỗi tầng), phân tầng theo trễ / đúng hẹn nói gì, và với từng câu hỏi kinh doanh thì nên tin
cách chia nào. Nêu ít nhất một giới hạn.